# Машинное обучение, ФКН ВШЭ

# Практическое задание 8. Обучение без учителя.

## Общая информация
Дата выдачи: 05.02.2026

Мягкий дедлайн: 19.02.2026 23:59 MSK

Жёсткий дедлайн: 26.02.2026 23:59 MSK

## Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи).

Сдавать задание после указанного срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке.

## Формат сдачи
Задания сдаются через систему anytask. Посылка должна содержать:
* Ноутбук homework-practice-08-Username.ipynb

Username — ваша фамилия на латинице

## О задании

В этом задании мы посмотрим на несколько алгоритмов кластеризации и применим их к географическим и текстовым данным. Также мы подробно остановимся на тематическом моделировании текстов, задаче обучения представлений и в каком-то смысле поработаем с semi-supervised learning.



In [ ]:
import os
from itertools import cycle

# import pandas as pd
import numpy as np
import optuna
import polars as pl
import sklearn

sklearn.set_config(transform_output="polars")

from sklearn.base import ClusterMixin
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE

np.random.seed(0xFFFFFFF)

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots

layout_dict = {
    "margin": {"l": 20, "r": 20, "t": 40, "b": 20},
    "width": 600,
    "height": 400,
    "paper_bgcolor": "LightSteelBlue",
    "title_font_size": 14,
    "xaxis_title_font_size": 12,
    "yaxis_title_font_size": 12,
}

In [ ]:
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
# kmeans = KMeans(n_clusters=3, random_state=0,).fit(X)
# calinski_harabasz_score(X, kmeans.labels_)
# silhouette_score(X, kmeans.labels_)
# davies_bouldin_score(X, kmeans.labels_)

In [ ]:
def wcss(X: pl.DataFrame, labels: np.array) -> float:
    return (
        X.with_columns(label=labels)
        .with_columns(
            pl.selectors.exclude("label")
            - pl.selectors.exclude("label").mean().over("label")
        )
        .select(pl.sum_horizontal(pl.exclude("label").pow(2)))
        .sum()
        .item()
    )

**Задание 0 (1e-100 балла)**. Какие ожидания от курса?

In [ ]:
# YOUR CODE HERE (ノಠ益ಠ)ノ彡┻━┻

## Часть 1. Кластеризация автобусных остановок

В этом задании мы сравним разные алгоритмы кластеризации для данных об автобусных остановках Москвы.

**Задание 1.1 (1 балл).** Реализуйте алгоритм спектральной кластеризации, который упоминался на лекции. Для этого разберитесь с кодом шаблона, данного ниже, и допишите недостающую функцию. Напомним, что для графа с матрицей смежности $W = \{w_{ij}\}_{i, j = 1 \dots \ell}$ лапласиан определяется как:

$$
L = D - W,
$$

где $D = \text{diag}(d_1, ..., d_{\ell}), d_i = \sum_{j=1}^{\ell} w_{ij}$.

In [ ]:
class GraphClustering(ClusterMixin):
    def __init__(self, n_clusters=8, n_components=None, **kwargs):
        """
        Spectral clustering algorithm
        param n_clusters: number of clusters to form
        param n_components: number of eigenvectors to use
        """

        if n_components is None:
            n_components = n_clusters

        self.n_components = n_components
        self.normalized = "normalized" in kwargs
        if "clustering_model" in kwargs:
            model = {"dbscan": DBSCAN, "kmeans": KMeans}[kwargs["clustering_model"]]
        else:
            model = KMeans
        if model == KMeans:
            self.model = model(n_clusters=n_clusters)
        elif model == DBSCAN:
            self.model = model(
                **{k: v for k, v in kwargs.items() if k in ["eps", "min_samples"]}
            )
        else:
            raise NotImplementedError

    def fit_predict(self, X, y=None):
        """
        Perform spectral clustering from graph adjacency matrix
        and return vertex labels.
        param X: (n_samples, n_samples) - graph adjacency matrix
        return: (n_samples, ) - vertex labels
        """
        if self.normalized:
            L = np.diag(X.sum(axis=1)) - X
        else:
            ## I - D^{-1/2}WD^{-1/2} # normalized Laplacian, may work better
            D_root = np.diag(np.pow(X.sum(axis=1), -0.5))
            L = np.eye(X.shape[0]) - (D_root @ X @ D_root)
        eigenvectors = self._generate_eigenvectors(L)
        labels = self.model.fit_predict(eigenvectors[:, 1:])
        return labels

    def _generate_eigenvectors(self, X):
        """
        Compute eigenvectors for spectral clustering
        param X: (n_samples, n_samples) - graph adjacency matrix
        return: (n_samples, n_components) - eigenvectors
        """
        evals, evecs = np.linalg.eigh(X)
        # drop those corresponding to zero eigenvalues..
        # mask = evals <= 1e-10
        # evals = evals[mask]
        # evecs = evecs[:, mask]
        order = sorted(range(len(evals)), key=np.abs(evals).__getitem__)
        return evecs[:, order[: self.n_components + 1]]

Перед тем, как переходить к следующему заданию, протестируйте свое решение.

In [ ]:
n_blocks, n_vertices = 10, 1000
block_vertices = n_vertices // n_blocks

X = np.zeros((n_vertices, n_vertices))
for i in range(0, n_vertices, block_vertices):
    X[i : i + block_vertices, i : i + block_vertices] = np.sqrt(i + 1)

graph_clustering = GraphClustering(n_clusters=n_blocks)
labels = graph_clustering.fit_predict(X)

true_labels = np.zeros(n_vertices, dtype=np.int32)
for i in range(0, n_vertices, block_vertices):
    true_labels[i : i + block_vertices] = labels[i]

assert labels.shape == (n_vertices,)
assert np.all(np.bincount(labels) == np.full(n_blocks, block_vertices))
assert np.all(labels == true_labels)

Теперь можем приступить к работе с реальными данными. Скачайте файл с данными об остановках общественного транспорта **в формате .xlsx** по [ссылке](https://disk.yandex.ru/i/J0HlcDXd_KyIeg). Для удобства визуализации мы будем работать только с остановками в ЦАО.

In [ ]:
DATA_DIR = "/data/ml-course-hse/ml2-2026-spring/homework-practice-08-unsupervised/"
DATA_PATH = os.path.join(DATA_DIR, "transport.xlsx")

if not os.path.exists(DATA_PATH):
    import requests

    public_key = "https://disk.yandex.ru/i/J0HlcDXd_KyIeg"

    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    response = requests.get(api_url, params={"public_key": public_key})
    response.raise_for_status()
    download_url = response.json()["href"]

    file_response = requests.get(download_url)
    file_response.raise_for_status()

    os.makedirs(DATA_DIR, exist_ok=True)
    with open(DATA_PATH, "wb") as f:
        f.write(file_response.content)

In [ ]:
schema_overrides = {
    "ID_en": pl.Int64,
    "Name_en": pl.String,
    "Longitude_WGS84_en": pl.Float64,
    "Latitude_WGS84_en": pl.Float64,
    "Street_en": pl.String,
    "AdmArea_en": pl.Categorical,
    "District_en": pl.Categorical,
    "RouteNumbers_en": pl.String,
    "StationName_en": pl.String,
    "Direction_en": pl.String,
    "Pavilion_en": pl.String,
    "OperatingOrgName_en": pl.String,
    "EntryState_en": pl.String,
    "global_id": pl.Int64,
    "geoData": pl.String,
}

data = pl.read_excel(DATA_PATH, engine="openpyxl", schema_overrides=schema_overrides)
data = data.filter(pl.col("AdmArea_en") == "Czentral`ny'j administrativny'j okrug")

data.head(6)

Воспользуемся библиотекой `folium` для визуализации данных.

In [ ]:
import folium

fig = folium.Figure(width=500, height=500)
mp = folium.Map([55.75215, 37.61819], zoom_start=12).add_to(fig)
for row in data.iter_rows(named=True):
    folium.Circle(
        [row["Latitude_WGS84_en"], row["Longitude_WGS84_en"]], radius=10
    ).add_to(mp)
fig

**Задание 1.2 (1 балл).** Попробуем построить граф, в котором вершинами будут остановки. Как вы уже могли заметить, для каждой остановки указаны номера маршрутов, проходящих через неё. Логично соединить ребрами соседние остановки каждого маршрута. Однако мы не знаем, в каком порядке автобусы объезжают остановки. Но мы можем применить эвристический алгоритм, который восстановит нам порядок маршрутов:

* Для каждого маршрута выделим список всех остановок, через которые он проходит.
* Выберем начальную остановку маршрута как точку, наиболее удаленную от всех остальных остановок этого маршрута.
* Каждую следующую точку маршрута будем выбирать как самую близкую из оставшихся точек (не включенных в маршрут ранее).

Фактически, у нас получается жадное решение задачи коммивояжера. Когда мы отсортировали маршруты, можем построить по ним граф. Будем строить его по таким правилам:

* Между двумя остановками будет ребро, если они являются соседними хотя бы на одном маршруте. Вес ребра равен числу маршрутов, на которых остановки являются соседними.
* В графе не будет петель (то есть у матрицы смежности будет нулевая диагональ).

Реализуйте предложенный способ построения графа. Для этого рекомендуется воспользоваться шаблонами, приведенными ниже.

In [ ]:
def get_routes(data):
    """
    Accumulate routes from raw data
    param data: pd.DataFrame - public transport stops data
    return: dict - unsorted stops ids for each route,
                   e.g. routes['A1'] = [356, 641, 190]
    """
    out = data.select(
        pl.col("ID_en"), pl.col("RouteNumbers_en").str.split("; ")
    ).explode("RouteNumbers_en", empty_as_null=False)

    out = out.join(
        data.select("ID_en", "Longitude_WGS84_en", "Latitude_WGS84_en"),
        how="left",
        on="ID_en",
    ).sort("RouteNumbers_en")
    return out


def guess_last_stop(routes: pl.DataFrame):
    return (
        routes.join(
            routes.group_by("RouteNumbers_en").agg(
                route_lgs=pl.col("Longitude_WGS84_en"),
                route_lts=pl.col("Latitude_WGS84_en"),
            ),
            how="left",
            on="RouteNumbers_en",
        )
        .select(
            pl.col("ID_en", "RouteNumbers_en"),
            dist=(pl.col("route_lgs") - pl.col("Longitude_WGS84_en"))
            .list.eval(pl.element() ** 2)
            .list.sum()
            + (pl.col("route_lts") - pl.col("Latitude_WGS84_en"))
            .list.eval(pl.element() ** 2)
            .list.sum(),
        )
        .group_by("RouteNumbers_en")
        .agg(pl.col("ID_en", "dist").max_by("dist"))
        .with_columns(order=pl.lit(0))
    )


def greedy_step(routes, t):
    return routes.update(
        routes.filter(pl.col("order").is_null())
        .join(
            routes.filter(pl.col("order") == t).select(
                "RouteNumbers_en",
                pl.col("Longitude_WGS84_en").alias("last_lg"),
                pl.col("Latitude_WGS84_en").alias("last_lt"),
            ),
            how="left",
            on="RouteNumbers_en",
        )
        .with_columns(
            dist=pl.col("Longitude_WGS84_en").sub("last_lg").pow(2)
            + pl.col("Latitude_WGS84_en").sub("last_lt").pow(2)
        )
        .group_by("RouteNumbers_en")
        .agg(pl.col("ID_en", "dist").min_by("dist"))
        .with_columns(order=t + 1),
        on=["RouteNumbers_en", "ID_en"],
    )


def sort_routes(routes):
    """
    Sort routes according to the proposed algorithm
    param data: pd.DataFrame - public transport stops data
    param routes: dict - unsorted stops ids for each route
    return: dict - sorted stops ids for each route
    """
    # routes = routes.with_columns(
    #     dist=pl.lit(None, dtype=pl.Float64), order=pl.lit(None, dtype=pl.Int64)
    # )
    routes = routes.join(
        guess_last_stop(routes), on=["ID_en", "RouteNumbers_en"], how="left"
    )

    # step = routes
    t = 0
    while routes["order"].is_null().any():
        routes = greedy_step(routes, t)
        t += 1
        if t > 100:
            print("Suspiciously long route")
            break
    # while len(step):
    #     step = greedy_step(step).with_columns(order=t)
    #     routes = routes.update(step, on=["ID_en", "RouteNumbers_en"])
    #     step = routes.filter(pl.col("order").is_null())
    #     t += 1

    return routes


def get_adjacency_matrix(sorted_routes):
    """
    Compute adjacency matrix for sorted routes
    param data: pd.DataFrame - public transport stops data
    param sorted_routes: dict - sorted stops ids for each route
    return: (n_samples, n_samples) - graph adjacency matrix
    """
    out = pl.concat(
        [
            sorted_routes.join(
                sorted_routes.select(
                    pl.col("RouteNumbers_en"),
                    pl.col("ID_en").alias("other_stop"),
                    order=pl.col("order") - 1,
                ),
                on=["RouteNumbers_en", "order"],
                how="inner",
            ),
            sorted_routes.join(
                sorted_routes.select(
                    pl.col("RouteNumbers_en"),
                    pl.col("ID_en").alias("other_stop"),
                    order=pl.col("order") + 1,
                ),
                on=["RouteNumbers_en", "order"],
                how="inner",
            ),
        ]
    )
    stop_ids = out["ID_en"].unique().sort()
    return (
        out.pivot(
            "other_stop",
            index="ID_en",
            values="ID_en",
            aggregate_function="len",
            on_columns=stop_ids,
        )
        .sort("ID_en")
        .with_columns(pl.selectors.exclude("ID_en").cast(pl.Int64))
    )


In [ ]:
routes = get_routes(data)
routes = sort_routes(routes)
adjacency_matrix = get_adjacency_matrix(routes)
assert (lambda x: x.T == x)(adjacency_matrix.drop("ID_en").to_numpy()).ravel().all()

Проверим, что маршруты получились адекватными. Для этого нарисуем их на карте.

In [ ]:
fig = folium.Figure(width=500, height=500)
mp = folium.Map([55.75215, 37.61819], zoom_start=12).add_to(fig)
for route_id in routes["RouteNumbers_en"].sample(5):
    # route_id = 'АН2'
    # route_id = 'А730'
    print(route_id)
    coords = (
        routes.filter(pl.col("RouteNumbers_en") == route_id)
        .sort("order")
        .select("Latitude_WGS84_en", "Longitude_WGS84_en")
        .to_numpy()
    )

    folium.vector_layers.PolyLine(coords).add_to(mp)

fig

**Задание 1.3 (0 баллов)**. Реализуйте функцию `draw_clustered_map`, которая рисует карту центра Москвы с кластерами остановок, раскрашенными в разные цвета.

In [ ]:
colors = [
    "#1f77b4",
    "#aec7e8",
    "#ff7f0e",
    "#ffbb78",
    "#2ca02c",
    "#98df8a",
    "#d62728",
    "#ff9896",
    "#9467bd",
    "#c5b0d5",
    "#8c564b",
    "#c49c94",
    "#e377c2",
    "#f7b6d2",
    "#7f7f7f",
    "#c7c7c7",
    "#bcbd22",
    "#dbdb8d",
    "#17becf",
    "#9edae5",
]


def draw_clustered_map(data, labels):
    """
    Create map with coloured clusters
    param data: pd.DataFrame - public transport stops data
    param labels: (n_samples, ) - cluster labels for each stop
    return: folium.Map - map with coloured clusters
    """

    color_dict = {x: c for x, c in zip(list(set(labels)), cycle(colors))}
    fig = folium.Figure(width=500, height=500)
    mp = folium.Map([55.75215, 37.61819], zoom_start=12).add_to(fig)
    for row, c in zip(data.iter_rows(named=True), map(color_dict.get, labels)):
        folium.Circle(
            [row["Latitude_WGS84_en"], row["Longitude_WGS84_en"]],
            radius=10,
            color=c,
            fill=True,
            fill_color=c,
            fill_opacity=0.75,
        ).add_to(mp)

    return fig

**Задание 1.4 (1.5 балла)**. Примените алгоритмы кластеризации и подберите гиперпараметры так, чтобы кластеры получились осмысленными (нет такого, что все принадлежит одному кластеру или большая часть точек это шум):
- `sklearn.clustering.KMeans` - `n_clusters`
- `sklearn.clustering.DBSCAN` - `eps`
- ваш `SpectralClustering` - `n_clusters` и `n_components`

Для ряда алгоритмов, в частности DBSCAN, подобрать оптимальные параметры может быть ой, как непросто. Чтобы помочь сохранить нервные клетки рекомендуется пользоваться метриками Intrinsic Evaluation для кластеризации. Для каждого алгоритма скорее всего придется брать разные метрики, а может быть оптимизировать сразу несколько.

Ваша задача - выбрать подходящую метрику, можно взять что-то из списка ниже, можно спросить у гугла. Применить GridSearch, `optuna` или `hyperopt` и желательно завернуть отбор параметров в функцию, пригодится далее

1. $
   \text{Calinski-Harabasz Index} \in [0, \inf] = \frac{\sum_{i=1}^k BCSS_i/(k-1)}{\sum_{i=1}^k WCSS_i/(n-k)} = \frac{\sum_{i=1}^k n_i \| \mathbf{c}_i - \mathbf{c} \|^2/(k-1)}{\sum_{i=1}^k \sum_{\mathbf{x} \in C_i} \|\mathbf{x} - \mathbf{c}_i\|^2/(n-k)} \rightarrow max
   $
   - Отношение 1) расстояния от центра кластера до центра всех данных к 2) расстоянию от центра кластера до точки
   - Чем однороднее и дальше друг от друга кластеры, тем лучше
2. $
   \text{Silhouette Score} \in [0, 1] = \sum_i \frac{b_i - a_i}{\max(a_i, b_i)} \rightarrow max
   $
   - $a_i$ - среднее расстояние до точек внутри своего кластера
   - $b_i$ - среднее расстояние до точек из ближайшего кластера
   - Если в своем кластере точки близко - кластер плотный, идеальный вариант, скор равен 1, если свой кластер совпадает с соседним - кластеры неразделимы, скор равен нулю
   - Чем однороднее и разделимее кластеры, тем лучше
3. $\text{Davies-Bouldin Index} \in [0, inf] = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} \left( \frac{WCSS_i + WCSS_j}{d(c_i, c_j)} \right) \rightarrow min $
   - Отношение внутрикластерных расстояний к межкластерному расстоянию между двумя центрами
   - Если кластеры далеко друг от друга, но при этом однородные, скор наименьший
3. $\text{Elbow Method}$ - последовательное сравнение $WCSS$ из пункта 1 для разного числа параметров. Если уменьшение меньше определенного порога - останавливаемся, считаем это значение оптимальным

In [ ]:
# step 1. Simply run everything with an arbitrary parameter

In [ ]:
# KMeans - n_clusters
# DBScan - eps (min_samples)
# Spectral Clustering (operates on adjacency_matrix)

In [ ]:
n_clusters = 10

km = KMeans(n_clusters=n_clusters, random_state=1)
labels = km.fit_predict(data["Longitude_WGS84_en", "Latitude_WGS84_en"])

draw_clustered_map(data, labels)  # this already pretty good actually

In [ ]:
eps = 3.2e-3
min_samples = 3


# {'dbscan_eps': 0.009000000000000001, 'dbscan_min_samples': 8}
# {'dbscan_eps': 0.004, 'dbscan_min_samples': 5}
# {'dbscan_eps': 0.008, 'dbscan_min_samples': 6}

# eps = 8e-3
# min_samples = 6


dbs = DBSCAN(eps=eps, min_samples=min_samples)
labels = dbs.fit_predict(data["Longitude_WGS84_en", "Latitude_WGS84_en"])

draw_clustered_map(data, labels)  # took some time to find good values


# [I 2026-08-27 23:34:23,855] Trial 30 finished with value: -0.47271394943698575 and parameters: {'dbscan_eps': 0.009000000000000001, 'dbscan_min_samples': 8}. Best is trial 30 with value: -0.47271394943698575.


In [ ]:
# Spectral Clustering
n_clusters = 15
n_components = 10

graph_clustering = GraphClustering(n_clusters=n_clusters, n_components=n_components)
labels = graph_clustering.fit_predict(adjacency_matrix.drop("ID_en").to_numpy())

draw_clustered_map(data, labels)


In [ ]:
# step 2. Optimize with Optuna systematically

In [ ]:
def clustering_objective_factory(model, metric):
    assert metric in (
        calinski_harabasz_score,
        davies_bouldin_score,
        silhouette_score,
    ), "Unsupported metric"
    assert model in (KMeans, DBSCAN, GraphClustering)

    def clustering_objective(trial: optuna.Trial):
        # большая часть точек это шум):
        # - `sklearn.clustering.KMeans` - `n_clusters`
        # - `sklearn.clustering.DBSCAN` - `eps`
        # - ваш `SpectralClustering` - `n_clusters` и `n_components`

        config = {}
        match model:
            case _ if model is KMeans:
                config["n_clusters"] = trial.suggest_int("n_clusters", low=8, high=64)
            case _ if model is DBSCAN:
                config["eps"] = trial.suggest_float("dbscan_eps", 1e-3, 1e-2, step=1e-3)
                config["min_samples"] = trial.suggest_int(
                    "dbscan_min_samples", low=3, high=8
                )
            case _ if model is GraphClustering:
                if trial.suggest_categorical("normalized", [True, False]):
                    config["normalized"] = True
                config["clustering_model"] = trial.suggest_categorical(
                    "clustering_model", ["dbscan", "kmeans"]
                )
                if config["clustering_model"] == "kmeans":
                    config["n_clusters"] = trial.suggest_int(
                        "n_clusters", low=8, high=32
                    )
                else:
                    config["eps"] = trial.suggest_float(
                        "dbscan_eps", 1e-3, 1e-2, step=1e-3
                    )
                    config["min_samples"] = trial.suggest_int(
                        "dbscan_min_samples", low=3, high=8
                    )

        clustering = model(**config)
        X = data["Longitude_WGS84_en", "Latitude_WGS84_en"]
        if model is GraphClustering:
            labels = clustering.fit_predict(adjacency_matrix.drop("ID_en").to_numpy())
        else:
            labels = clustering.fit_predict(X)

        if 1 < np.unique(labels).size:
            trial.set_user_attr("Calinski Harabasz", calinski_harabasz_score(X, labels))
            trial.set_user_attr("Silhouette Score", silhouette_score(X, labels))
            trial.set_user_attr("Davies Bouldin", davies_bouldin_score(X, labels))

            return metric(data["Longitude_WGS84_en", "Latitude_WGS84_en"], labels) * (
                -1 if metric is davies_bouldin_score else 1
            )

        return -np.inf

    return clustering_objective

In [ ]:
ws = []
for n_clusters in range(8, 64):
    km = KMeans(n_clusters=n_clusters)
    labels = km.fit_predict(data["Longitude_WGS84_en", "Latitude_WGS84_en"])
    ws.append(wcss(data["Longitude_WGS84_en", "Latitude_WGS84_en"], labels))

# 16 clusters is about right..

fig = px.line(pl.DataFrame({"wcss": ws}).with_row_index("ix"), "ix", "wcss")

fig.update_layout(**layout_dict)
fig.show()

In [ ]:
study_name = "kmeans"
clustering_objective = clustering_objective_factory(KMeans, davies_bouldin_score)

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study.optimize(
    clustering_objective,
    n_trials=60,
)
print(study.best_params)

In [ ]:
ws = []
epss = np.linspace(1e-4, 1e-2, 20)
for eps in epss:
    dbs = DBSCAN(eps=eps, min_samples=3)
    labels = dbs.fit_predict(data["Longitude_WGS84_en", "Latitude_WGS84_en"])
    ws.append(wcss(data["Longitude_WGS84_en", "Latitude_WGS84_en"], labels))

## 3.2e-3
fig = px.line(
    pl.DataFrame({"wcss": ws, "eps": epss}).with_row_index("ix"), "eps", "wcss"
)

fig.update_layout(**layout_dict)
fig.show()

In [ ]:
ws = []

eps = 3.2e-3
mss = [3, 4, 5, 6, 7, 8]
for ms in mss:
    dbs = DBSCAN(eps=eps, min_samples=ms)
    labels = dbs.fit_predict(data["Longitude_WGS84_en", "Latitude_WGS84_en"])
    ws.append(wcss(data["Longitude_WGS84_en", "Latitude_WGS84_en"], labels))

# 16 clusters is about right..
fig = px.line(pl.DataFrame({"wcss": ws, "min_samples": mss}), "min_samples", "wcss")

fig.update_layout(**layout_dict)
fig.show()

In [ ]:
study_name = "dbscan"
clustering_objective = clustering_objective_factory(DBSCAN, silhouette_score)

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study.optimize(
    clustering_objective,
    n_trials=60,
)
print(study.best_params)

In [ ]:
study_name = "graph_clustering"
clustering_objective = clustering_objective_factory(
    GraphClustering, davies_bouldin_score
)

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study.optimize(
    clustering_objective,
    n_trials=60,
)
print(study.best_params)

In [ ]:
# Spectral Clustering
graph_clustering = GraphClustering(clustering_model="dbscan", eps=0.009, min_samples=7)
labels = graph_clustering.fit_predict(adjacency_matrix.drop("ID_en").to_numpy())

draw_clustered_map(data, labels)

Визуализируйте результат кластеризации с помощью функции `draw_clustered_map`.

**Вопрос:** Чем отличаются разбиения на кластеры, получаемые разными алгоритмами? Какие плюсы и минусы есть у каждого алгоритма? Какой алгоритм кажется вам наиболее подходящим для кластеризации остановок?

**Ответ:** All metrics do not perform spectacularly, but interestingly GraphClustering has an entirely different notion of similarity based on route network rather than geographic proximity

## Часть 2. Тематическое моделирование текстов

В этой части мы познакомимся с одной из самых популярных задач обучения без учителя &mdash; с задачей тематического моделирования текстов. Допустим, нам доступна некоторая коллекция документов без разметки, и мы хотим автоматически выделить несколько тем, которые встречаются в документах, а также присвоить каждому документу одну (или несколько) тем. Фактически, мы будем решать задачу, похожую на кластеризацию текстов: отличие в том, что нас будет интересовать не только разбиение текстов на группы, но и выделение ключевых слов, определяющих каждую тему.

Мы будем работать с новостными статьями BBC за 2004-2005 годы. Скачайте данные по [ссылке](https://www.kaggle.com/hgultekin/bbcnewsarchive).

In [ ]:
!export KAGGLE_API_TOKEN=$(cat /data/config/.kaggle/token) && /opt/venv/bin/kaggle datasets list -s BBC-News

In [ ]:
!mkdir -p /data/ml-course-hse/ml2-2026-spring/homework-practice-08-unsupervised && export KAGGLE_API_TOKEN=$(cat /data/config/.kaggle/token) && /opt/venv/bin/kaggle datasets download hgultekin/bbcnewsarchive -p /data/ml-course-hse/ml2-2026-spring/homework-practice-08-unsupervised --unzip

In [ ]:
data = pl.read_csv(
    "/data/ml-course-hse/ml2-2026-spring/homework-practice-08-unsupervised/bbc-news-data.csv",
    separator="\t",
    quote_char=None,
)
data.sample(5)

Как вы могли заметить, данные уже содержат разметку по тематике (колонка category). В этой части мы забудем, что она есть, и будем работать только с текстовыми данными. Проведем предобработку текста, состоящую из следующих пунктов:

* Объединим заголовок и содержание статьи в одно поле.
* Приведем текст к нижнему регистру, разобьем его на токены.
* Оставим только буквенные слова (удалив, таким образом, пунктуацию и числа).
* Применим лемматизацию.
* Удалим стоп-слова.


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("stopwords")

stop_words = set(stopwords.words("english") + ["ha", "wa", "say", "said"])
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess(text):
    text = list(filter(str.isalpha, word_tokenize(text.lower())))
    text = [lemmatizer.lemmatize(word) for word in text]
    text = [word for word in text if word not in stop_words]
    return " ".join(text)

In [ ]:
data = data.with_columns(raw_text=pl.col("title") + pl.col("content")).with_columns(
    text=pl.col("raw_text").map_elements(preprocess, pl.String)
)

Для визуализации частот слов в текстах мы будем использовать [облака тегов](https://en.wikipedia.org/wiki/Tag_cloud).

In [ ]:
# !uv add wordcloud

In [ ]:
from wordcloud import WordCloud


def draw_wordcloud(texts, max_words=1000, width=1000, height=500):
    wordcloud = WordCloud(
        background_color="white", max_words=max_words, width=width, height=height
    )

    joint_texts = " ".join(list(texts))
    wordcloud.generate(joint_texts)
    return wordcloud.to_image()

In [ ]:
draw_wordcloud(data["text"])

**Задание 2.1 (1 балл).** Обучите алгоритм K-Means на tf-idf представлениях текстов. При обучении tf-idf векторайзера рекомендуется отбрасывать редко встречающиеся слова, а также воздержитесь от использования N-грамм. Возьмите не очень большое число кластеров, чтобы было удобно интерпретировать получившиеся темы (например, `n_clusters` = 8). Постройте облака тегов для текстов из разных кластеров.

In [ ]:
tfidf_vec = TfidfVectorizer(min_df=10, max_features=500)
tfidf_vec.fit(data["text"])
df = pl.DataFrame(
    tfidf_vec.transform(data["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)

In [ ]:
n_clusters = 8

km = KMeans(n_clusters=n_clusters, random_state=42)
data = data.with_columns(cluster_ix=km.fit_predict(df).astype("int64"))

In [ ]:
k_largest = 5
arg_words = np.argpartition(-np.abs(km.cluster_centers_), axis=-1, kth=k_largest)[
    :, :k_largest
]

categories = data["category"].unique().sort()

for row in (
    data.pivot(
        on="category",
        on_columns=categories,
        index="cluster_ix",
        values="cluster_ix",
        aggregate_function=pl.element().len(),
    )
    .with_columns(
        most_common=pl.lit(categories).gather(
            pl.concat_list(pl.exclude("cluster_ix")).list.arg_max()
        )
    )
    .sort("cluster_ix")
    .iter_rows(named=True)
):
    print(f"Cluster {row['cluster_ix']}, most common category: {row['most_common']}")
    print(
        f"Largest words: {tfidf_vec.get_feature_names_out()[arg_words[row['cluster_ix']]]}"
    )
    display(
        draw_wordcloud(data.filter(pl.col("cluster_ix") == row["cluster_ix"])["text"])
    )

Получились ли темы интерпретируемыми? Попробуйте озаглавить каждую тему.

**Ответ:**

**Задание 2.2 (0.5 балла).** Попробуем другой способ выделить ключевые слова для каждой темы. Помимо непосредственного разбиения объектов алгоритм K-Means получает центр каждого кластера. Попробуйте взять центры кластеров и посмотреть на слова, для которых значения соответствующих им признаков максимальны.

Согласуются ли полученные слова с облаками тегов из прошлого задания?

**Ответ:** Very reasonably well

**Задание 2.3 (1.5 балла).** В первой части мы сравнили три разных алгоритма кластеризации на географических данных. Проделаем то же самое для текстовых данных (в качестве признакого описания снова используем tf-idf). Получите три разбиения на кластеры с помощью алгоритмов K-Means, DBSCAN и спектральной кластеризации (на этот раз воспользуйтесь реализацией из `sklearn`). Для K-Means и спектральной кластеризации возьмите одинаковое небольшое число кластеров, подберите параметр `eps` метода DBSCAN так, чтобы получить приблизительно такое же число кластеров (подумайте, как это можно сделать)

Далее, обучите двухмерные t-SNE представления над tf-idf признаками текстов. Визуализируйте эти представления для каждого алгоритма, раскрасив каждый кластер своим цветом. Лучше всего расположить визуализации на одном графике на трех разных сабплотах. Не забудьте, что DBSCAN помечает некоторые точки как шумовые (можно раскрасить их в отдельный цвет).

In [ ]:
# Optimize n_clusters for K-means using Davies Bouldin
# davies_bouldin_score(X, kmeans.labels_)

# km = KMeans(n_clusters=n_clusters, random_state=42)
# data = data.with_columns(cluster_ix=km.fit_predict(df).astype("int64"))

elbow = pl.DataFrame(
    schema={"n_clusters": pl.Int64, "db_score": pl.Float64, "wcss": pl.Float64}
)

for n_clusters in range(8, 128, 2):
    km_ = KMeans(n_clusters=n_clusters, random_state=42)
    km_.fit(df)
    elbow.extend(
        pl.DataFrame(
            {
                "n_clusters": n_clusters,
                "db_score": davies_bouldin_score(df, km_.labels_),
                "wcss": wcss(df, km_.labels_),
            }
        )
    )

elbow = elbow.with_columns(
    db_elbow=pl.col("db_score").diff(-1) / pl.col("db_score").diff(1).neg(),
    wcss_elbow=pl.col("wcss").diff(-1) / pl.col("wcss").diff(1).neg(),
)

In [ ]:
# db_score -> 30 clusters
fig = px.line(
    elbow,
    x="n_clusters",
    y=["db_score"],
)

fig.update_layout(**layout_dict)

In [ ]:
elbow.filter(pl.col("n_clusters") == 30)

In [ ]:
km = KMeans(n_clusters=30, random_state=42)
km.fit(df)

{
    "db_score": davies_bouldin_score(df, km.labels_),
    "wcss": wcss(df, km.labels_),
}

In [ ]:
## In practice, built in objectives lead to "uninteresting" dbscan results
# import optuna

# def dbscan_objective(trial: optuna.Trial):
#     config = {}
#     config["eps"] = trial.suggest_float("eps", 1e-1, 3e0, step=5e-2)
#     config["min_samples"] = trial.suggest_int("min_samples", low=3, high=16)

#     model = DBSCAN(**config)
#     labels = model.fit_predict(df)

#     if 1 < np.unique(labels).size:
#         trial.set_user_attr("Calinski Harabasz", calinski_harabasz_score(df, labels))
#         trial.set_user_attr("Silhouette Score", silhouette_score(df, labels))
#         trial.set_user_attr("Davies Bouldin", davies_bouldin_score(df, labels))

#         return davies_bouldin_score(df, labels)

#     return np.inf

# study_name = "dbscan_bbc_news"

# sampler = optuna.samplers.TPESampler(
#     seed=10
# )  # Make the sampler behave in a deterministic way.
# study = optuna.create_study(
#     study_name=study_name,
#     sampler=sampler,
#     pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
#     # storage=f"sqlite:////data/{study_name}.db",
#     direction="minimize",
#     # load_if_exists=True,
# )
# study.optimize(
#     dbscan_objective,
#     n_trials=200,
# )
# print(study.best_params)

In [ ]:
# db_ = DBSCAN(eps=0.99755, min_samples=8)
# db_ = DBSCAN(eps=0.993195, min_samples=7)
db = DBSCAN(eps=0.97993, min_samples=6)
labels = db.fit_predict(df)
# print(davies_bouldin_score(df, labels))
pl.Series(labels).value_counts().sort(by="count")

In [ ]:
spc = SpectralClustering(n_clusters=30, n_components=8, random_state=32, affinity="rbf")
labels = spc.fit_predict(df)
pl.Series(labels).value_counts().sort(by="count")

In [ ]:
perplexity = 30
tsne = TSNE(n_components=2, perplexity=perplexity)

df_tsne = tsne.fit_transform(df)
df_tsne = df_tsne.with_columns(
    cluster_km=km.predict(df).astype("str"),
    cluster_db=db.fit_predict(df).astype("str"),
    cluster_sp=spc.fit_predict(df).astype("str"),
)

In [ ]:
colors = [
    "#1f77b4",
    "#aec7e8",
    "#ff7f0e",
    "#ffbb78",
    "#2ca02c",
    "#98df8a",
    "#d62728",
    "#ff9896",
    "#9467bd",
    "#c5b0d5",
    "#8c564b",
    "#c49c94",
    "#e377c2",
    "#f7b6d2",
    "#7f7f7f",
    "#c7c7c7",
    "#bcbd22",
    "#dbdb8d",
    "#17becf",
    "#9edae5",
]


cluster_cols = ["cluster_km", "cluster_db", "cluster_sp"]

fig = make_subplots(rows=1, cols=3, subplot_titles=cluster_cols)

for i, col in enumerate(cluster_cols, start=1):
    labels = df_tsne[col].unique().sort().cast(pl.String).to_list()
    color_dict = dict(zip(labels, cycle(colors)))
    sub = px.scatter(
        df_tsne,
        "tsne0",
        "tsne1",
        color=col,
        color_discrete_map=color_dict,
        category_orders={col: labels},
    )
    for trace in sub.data:
        fig.add_trace(trace.update(showlegend=False), row=1, col=i)

fig.update_layout(**layout_dict)
fig.update_layout(width=1500)


Прокомментируйте получившиеся результаты. Какой баланс кластеров получился у разных методов? Соотносятся ли визуализации для текстов с визуализациями для географических данных?

**Ответ:**

**Задание 2.4 (1.5 балла).** Обучите модель латентного размещения Дирихле. Не забудьте, что она работает с мешком слов, а не с tf-idf признаками. Придумайте, как превратить распределение тем для текста в номер его кластера. Возьмите параметр `n_components` в 2-3 раза больше, чем число кластеров для K-Means.

In [ ]:
cnt_vec = CountVectorizer(max_features=500, min_df=0.05, max_df=0.8)

cnt_vec.fit(data["text"])
dfc = pl.DataFrame(
    cnt_vec.transform(data["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)

In [ ]:
lda = LatentDirichletAllocation(n_components=64, random_state=31)
dfc = lda.fit_transform(dfc)
dfc = dfc.with_columns(
    latent_mode=pl.concat_list(pl.all()).list.arg_max(), category=data["category"]
)

In [ ]:
k_largest = 5
categories = data["category"].unique().sort()

arg_words = np.argpartition(-np.abs(lda.components_), axis=-1, kth=k_largest)[
    :, :k_largest
]

for row in (
    dfc.pivot(
        on="category",
        on_columns=categories,
        index="latent_mode",
        values="latent_mode",
        aggregate_function=pl.element().len(),
    )
    .with_columns(
        most_common=pl.lit(categories).gather(
            pl.concat_list(pl.exclude("latent_mode")).list.arg_max()
        )
    )
    .sort("latent_mode")
    .iter_rows(named=True)
):
    print(
        f"Latent compoent {row['latent_mode']}, most common category: {row['most_common']}"
    )
    print(
        f"Largest words: {cnt_vec.get_feature_names_out()[arg_words[row['latent_mode']]]}"
    )
    if row["most_common"] == "sport":
        display(
            draw_wordcloud(
                data.filter(dfc["latent_mode"] == row["latent_mode"])["text"]
            )
        )

Получились ли темы более узкими от такого нововведения? Постройте облака тегов для нескольких наиболее удачных тем.

**Ответ:**

## Часть 3. Transfer learning и Self-Supervised Learning для задачи классификации текстов

**Задание 3.1 (0.5 балла).** Вспомним, что у нас есть разметка для тематик статей. Попробуем обучить классификатор поверх unsupervised-представлений для текстов. Рассмотрите три модели:

* Логистическая регрессия на tf-idf признаках
* K-Means на tf-idf признаках + логистическая регрессия на расстояниях до центров кластеров
* Латентное размещение Дирихле + логистическая регрессия на вероятностях тем

Разделите выборку на обучающую и тестовую, замерьте accuracy на обоих выборках для всех трех моделей. Параметры всех моделей возьмите равными значениям по умолчанию.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data_train, data_test = train_test_split(
    data, test_size=0.25, random_state=11, shuffle=True
)

In [ ]:
# logistic regression
tfidf_vec = TfidfVectorizer(min_df=10, max_features=500)
tfidf_vec.fit(data_train["text"])
df_train1 = pl.DataFrame(
    tfidf_vec.transform(data_train["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)
df_test1 = pl.DataFrame(
    tfidf_vec.transform(data_test["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)

In [ ]:
reg1 = LogisticRegression()
reg1.fit(X=df_train1, y=data_train["category"])

print(
    f"Logistic regression on TF-IDF train accuracy: {accuracy_score(data_train['category'], y_pred=reg1.predict(df_train1)):.3f}"
)
print(
    f"Logistic regression on TF-IDF test accuracy: {accuracy_score(data_test['category'], y_pred=reg1.predict(df_test1)):.3f}"
)


In [ ]:
km = KMeans(n_clusters=30, random_state=42)
km.fit(df_train1)

reg2 = LogisticRegression(max_iter=1000)
reg2.fit(X=km.transform(df_train1), y=data_train["category"])

print(
    f"Logistic regression on KM cluster distances train accuracy: {accuracy_score(data_train['category'], y_pred=reg2.predict(km.transform(df_train1))):.3f}"
)
print(
    f"Logistic regression on KM cluster distances test accuracy: {accuracy_score(data_test['category'], y_pred=reg2.predict(km.transform(df_test1))):.3f}"
)

In [ ]:
cnt_vec = CountVectorizer(max_features=500, min_df=0.05, max_df=0.8)
cnt_vec.fit(data_train["text"])
df_train2 = pl.DataFrame(
    cnt_vec.transform(data_train["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)
df_test2 = pl.DataFrame(
    cnt_vec.transform(data_test["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)

In [ ]:
lda = LatentDirichletAllocation(n_components=64, random_state=31)
lda.fit(df_train2)

reg3 = LogisticRegression(max_iter=1000)
reg3.fit(X=lda.transform(df_train2), y=data_train["category"])

print(
    f"Logistic regression on LDA components train accuracy: {accuracy_score(data_train['category'], y_pred=reg3.predict(lda.transform(df_train2))):.3f}"
)
print(
    f"Logistic regression on LDA components test accuracy: {accuracy_score(data_test['category'], y_pred=reg3.predict(lda.transform(df_test2))):.3f}"
)


У какой модели получилось лучшее качество? С чем это связано?

**Ответ:** Logistic Regression is by far the best: it is the least restricted model with 500 features

**Задание 3.2 (1.5 балла).** Теперь просимулируем ситуацию слабой разметки, которая часто встречается в реальных данных. Разделим обучающую выборку в пропорции 5:65:30. Будем называть части, соответственно, размеченный трейн, неразмеченный трейн и валидация.

Все unsupervised-алгоритмы (векторайзеры и алгоритмы кластеризации) запускайте на всем трейне целиком (размеченном и неразмеченном, суммарно 70%), а итоговый классификатор обучайте только на размеченном трейне (5%). Подберите гиперпараметры моделей по качеству на валидации (30%), а затем оцените качество на тестовой выборке (которая осталась от прошлого задания). Не скромничайте при подборе числа кластеров, сейчас нас интересует не интерпретируемое разбиение выборки, а итоговое качество классификации.

In [ ]:
data_train_, data_val = train_test_split(
    data_train, test_size=0.3, random_state=11, shuffle=True
)
data_train_wl, data_train_nl = train_test_split(
    data_train_, test_size=65 / 70, random_state=11, shuffle=True
)

# logistic regression
tfidf_vec = TfidfVectorizer(min_df=10, max_features=500)
tfidf_vec.fit(data_train_["text"])
df_train_wl = pl.DataFrame(
    tfidf_vec.transform(data_train_wl["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)
df_val = pl.DataFrame(
    tfidf_vec.transform(data_val["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)
df_train_ = pl.DataFrame(
    tfidf_vec.transform(data_train_["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)
df_test = pl.DataFrame(
    tfidf_vec.transform(data_test["text"]).toarray(),
    schema={k: pl.Float64 for k in tfidf_vec.get_feature_names_out()},
)


cnt_vec = CountVectorizer(max_features=500, min_df=0.05, max_df=0.8)
cnt_vec.fit(data_train_["text"])
df_train_wl2 = pl.DataFrame(
    cnt_vec.transform(data_train_wl["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)
df_val2 = pl.DataFrame(
    cnt_vec.transform(data_val["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)
df_train2_ = pl.DataFrame(
    cnt_vec.transform(data_train_["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)
df_test2 = pl.DataFrame(
    cnt_vec.transform(data_test["text"]).toarray(),
    schema={k: pl.Float64 for k in cnt_vec.get_feature_names_out()},
)

In [ ]:
def objective_reg1(trial: optuna.Trial) -> float:
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0, step=0.05)
    C = trial.suggest_float("C", 1e-2, 1e3, log=True)
    reg = LogisticRegression(solver="saga", l1_ratio=l1_ratio, C=C, max_iter=1000)
    reg.fit(df_train_wl, data_train_wl["category"])
    return accuracy_score(data_val["category"], reg.predict(df_val))

In [ ]:
study_name = "regression1_semantic"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study1 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study1.optimize(
    objective_reg1,
    n_trials=100,
)
print(study1.best_params)

In [ ]:
def objective_reg2(trial: optuna.Trial) -> float:
    n_clusters = trial.suggest_int("n_clusters", 16, 128, step=8)
    km = KMeans(n_clusters=n_clusters, random_state=42)
    km.fit(df_train_)
    C = trial.suggest_float("C", 1e-2, 1e3, log=True)
    reg = LogisticRegression(l1_ratio=0.0, C=C, max_iter=1000)
    reg.fit(X=km.transform(df_train_wl), y=data_train_wl["category"])
    return accuracy_score(data_val["category"], reg.predict(km.transform(df_val)))

In [ ]:
study_name = "regression2_semantic"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study2 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study2.optimize(
    objective_reg2,
    n_trials=100,
)
print(study2.best_params)

In [ ]:
def objective_reg3(trial: optuna.Trial) -> float:
    n_components = trial.suggest_int("n_components", 16, 256, step=4)
    doc_topic_mult = trial.suggest_float("doc_topic_mult", 0.25, 2.5, step=0.25)
    topic_word_mult = trial.suggest_float("topic_word_mult", 0.25, 2.5, step=0.25)
    lda = LatentDirichletAllocation(
        n_components=n_components,
        doc_topic_prior=doc_topic_mult / n_components,
        topic_word_prior=topic_word_mult / n_components,
        random_state=31,
    )
    lda.fit(df_train2_)
    C = trial.suggest_float("C", 1e-2, 1e3, log=True)
    reg = LogisticRegression(l1_ratio=0.0, C=C, max_iter=1000)
    reg.fit(X=lda.transform(df_train_wl2), y=data_train_wl["category"])
    return accuracy_score(data_val["category"], reg.predict(lda.transform(df_val2)))

In [ ]:
study_name = "regression3_semantic"

sampler = optuna.samplers.TPESampler(
    seed=10
)  # Make the sampler behave in a deterministic way.
study3 = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    pruner=optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=5),
    # storage=f"sqlite:////data/{study_name}.db",
    direction="maximize",
    # load_if_exists=True,
)
study3.optimize(
    objective_reg3,
    n_trials=100,
)
print(study3.best_params)

In [ ]:
reg1 = LogisticRegression(**study1.best_params)
reg1.fit(X=df_train_wl, y=data_train_wl["category"])

print(
    f"Logistic regression on TF-IDF val accuracy: {accuracy_score(data_val['category'], y_pred=reg1.predict(df_val)):.3f}"
)
print(
    f"Logistic regression on TF-IDF test accuracy: {accuracy_score(data_test['category'], y_pred=reg1.predict(df_test)):.3f}"
)

In [ ]:
km = KMeans(n_clusters=study2.best_params["n_clusters"], random_state=42)
km.fit(df_train_)

reg2 = LogisticRegression(C=study2.best_params["C"], max_iter=1000)
reg2.fit(X=km.transform(df_train_wl), y=data_train_wl["category"])

print(
    f"Logistic regression on KM cluster distances val accuracy: {accuracy_score(data_val['category'], y_pred=reg2.predict(km.transform(df_val))):.3f}"
)
print(
    f"Logistic regression on KM cluster distances test accuracy: {accuracy_score(data_test['category'], y_pred=reg2.predict(km.transform(df_test))):.3f}"
)

In [ ]:
n_components = study3.best_params["n_components"]
lda = LatentDirichletAllocation(
    n_components=n_components,
    doc_topic_prior=study3.best_params["doc_topic_mult"] / n_components,
    topic_word_prior=study3.best_params["topic_word_mult"] / n_components,
    random_state=31,
)
lda.fit(df_train2_)

reg3 = LogisticRegression(C=study3.best_params["C"], max_iter=1000)
reg3.fit(X=lda.transform(df_train_wl2), y=data_train_wl["category"])

print(
    f"Logistic regression on LDA components val accuracy: {accuracy_score(data_val['category'], y_pred=reg3.predict(lda.transform(df_val2))):.3f}"
)
print(
    f"Logistic regression on LDA components test accuracy: {accuracy_score(data_test['category'], y_pred=reg3.predict(lda.transform(df_test2))):.3f}"
)

Как изменились результаты по сравнению с обучением на полной разметке? Сделайте выводы.

**Ответ:** KMeans on distances to cluster centers wins. Interestingly, it has weaker regularization and more components vs best LDA choice.

## Бонус

**Задание 4 (0.5 балла)**. Разберитесь с semi-supervised методами, которые реализованы в `sklearn` и примените их к заданию 3.2. Получилось ли добиться лучшего качества? Сделайте выводы.

In [ ]:
# YOUR CODE HERE ‿︵‿︵ヽ(°□° )ノ︵‿︵‿

**Задание 5 (1 балл)**. На занятиях мы обсуждали, что метрика [BCubed](https://www.researchgate.net/profile/Julio-Gonzalo-2/publication/225548032_Amigo_E_Gonzalo_J_Artiles_J_et_alA_comparison_of_extrinsic_clustering_evaluation_metrics_based_on_formal_constraints_Inform_Retriev_12461-486/links/0c96052138dbb99740000000/Amigo-E-Gonzalo-J-Artiles-J-et-alA-comparison-of-extrinsic-clustering-evaluation-metrics-based-on-formal-constraints-Inform-Retriev-12461-486.pdf) хорошо подходит для сравнения алгоритмов кластеризации, если нам известно настоящее разделение на кластеры (gold standard). Реализуйте подсчет метрики BCubed и сравните несколько алгоритмов кластеризации на текстовых данных из основного задания. В качестве gold standard используйте разметку category.

In [ ]:
# YOUR CODE HERE ‿︵‿︵ヽ(°□° )ノ︵‿︵‿

**Задание 6 (2 баллa)**. Спектральная кластеризация, по сути, является обычной кластеризацией KMeans поверх эмбеддингов объектов, которые получаются из лапласиана графа. А что, если мы попробуем построить эмбеддинги каким-нибудь другим способом? В этом задании мы предлагаем вам проявить немного фантазии. Возьмите какие-нибудь данные высокой размерности, чтобы задача обучения эмбеддингов имела смысл (например, картинки или тексты, желательно выбрать что-нибудь оригинальное). Придумайте или найдите какой-нибудь метод обучения эмбеддингов, примените его к данным и кластеризуйте полученные представления. Если чувствуете в себе достаточно силы, можете попробовать что-нибудь нейросетевое. Сравните ваш подход с базовыми алгоритмами кластеризации, которое мы рассмотрели в основном задании, не забывайте про визуализации! Ключевые слова для вдохновения: ***KernelPCA***, ***UMAP***, ***autoencoders***, ***gensim***.

In [ ]:
# YOUR CODE HERE ‿︵‿︵ヽ(°□° )ノ︵‿︵‿

**Задание 7 (1 балл)**. Наконец, ставший ежегодной традицией социализационный бонус. Мы поощряем не только предметное, но и духовное развитие. Поэтому, чтобы заработать балл за это задание, сходите на какую-нибудь выставку, в музей, театр, или наконец в церковь, напишите небольшой отчетик о ваших впечатлениях и добавьте фотопруфы в ноутбук при сдаче. Можете объединиться с одногруппниками/однокурсниками, а также пригласить ассистентов/преподавателей, они тоже будут рады выбраться куда-нибудь. Для вдохновения приведем ссылку на актуальные выставки [новой](https://www.youtube.com/watch?v=dQw4w9WgXcQ&ab) и [старой Третьяковки](https://www.youtube.com/watch?v=X1-MXdyThJ0) (но совсем не обязательно посещать именно их).

In [ ]:
# YOUR CODE HERE (ノಠ益ಠ)ノ彡┻━┻